# Reinforcement Learning in GridWorld

In this notebook we build a **GridWorld environment** and train an agent using **Q-Learning**.

Given an $n \times n$ grid with a target `T`, the goal of agent `A` is to learn a policy that reaches `T` as efficiently as possible. We explore three progressively more challenging variants:

1. **Standard GridWorld** — the agent knows its own position and the target's position.
2. **Blind GridWorld** — the target position is hidden from the agent.
3. **Toroidal GridWorld** — the grid wraps around like a torus, removing boundaries.

## Imports

We import:

- `numpy` — array operations and linear algebra
- `gymnasium` — standard interface for defining RL environments
- `defaultdict` — efficient storage for the Q-table (maps state–action pairs to values)
- `sys` and `time` — used for terminal-based rendering

In [ ]:
import numpy as np
import random
import gymnasium as gym
from collections import defaultdict
import sys, time

## Distance Functions

These functions measure the distance between two grid positions. We will use them to populate the `info` dictionary returned by the environment.

- **Euclidean norm** (`p=2`): straight-line distance, the default.
- **Manhattan norm** (`p=1`): sum of absolute coordinate differences, useful for grid navigation.

In [ ]:
def grid_distance(v, w, p=2):
    return np.linalg.norm(v - w, ord=p)

# Euclidean distance between two vectors.
# The parameter p controls the norm:
# p = 2 -> Euclidean norm
# p = 1 -> Manhattan norm


# Part 1 — Standard GridWorld

In this first variant the agent knows both its own position and the target's position at every step.

Key design choices:
- The target is re-randomised at the start of each episode, so the agent must learn a general policy, not a fixed path.
- The observation space is a dictionary with two `Box` entries (`agent` and `target`).
- The reward signal is `+1` on reaching the target and `−0.1` per step otherwise, encouraging the agent to find short paths.

In [ ]:
# Define GridWorld and GridAgent

# GridWorld environment definition.
# The environment is a square grid where:
# - A = agent
# - T = target
# The goal is to learn a policy that reaches the target efficiently.

class GridWorldEnv(gym.Env):
    metadata = {"render_modes": ["ansi"], "render_fps": 4}
    
    def __init__(self, size):
        # The size of the square grid 
        self.size = size

        # Initialize positions - will be set randomly in reset()
        # Using -1,-1 as "uninitialized" state
        self._agent_location = np.array([-1, -1], dtype=np.int32)
        self._target_location = np.array([-1, -1], dtype=np.int32)

        # Define what the agent can observe
        self.observation_space = gym.spaces.Dict(
                {
                    "agent" : gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),   # [x, y] coordinates
                    "target": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),  # [x, y] coordinates
                }
        )

        # Define what actions are available (4 directions)
        self.action_space = gym.spaces.Discrete(4)

        self.render_mode = "ansi"

        # Map action numbers to actual movements on the grid
        self._action_to_direction = {
                0: np.array([1, 0]),   # Move right (positive x)
                1: np.array([0, 1]),   # Move up (positive y)
                2: np.array([-1, 0]),  # Move left (negative x)
                3: np.array([0, -1]),  # Move down (negative y)
        }

    def _get_obs(self):
        """Convert internal state to observation format.
        Returns:
        dict: Observation with agent and target positions
        """
        return {"agent": self._agent_location, "target": self._target_location}

    
    def _get_info(self):
        distance = np.linalg.norm(self._agent_location - self._target_location,ord=2)
        return {"distance": distance}


    def reset(self, seed=None, options=None):
        """Start a new episode.
        Args:
        seed: Random seed for reproducible episodes
        options: Additional configuration (unused in this example)

        Returns:
        tuple: (observation, info) for the initial state
        """
        # IMPORTANT: Must call this first to seed the random number generator
        super().reset(seed=seed)

        # Randomly place the agent anywhere on the grid
        self._agent_location = self.np_random.integers(0, self.size, size=2, dtype=int)
            
        # Randomly place target, ensuring it's different from agent position
        self._target_location = self._agent_location
        while np.array_equal(self._target_location, self._agent_location):
            self._target_location = self.np_random.integers(0, self.size, size=2, dtype=int)
        observation = self._get_obs()
        info = self._get_info()
        return observation, info
	
    def step(self, action):
        """Execute one timestep within the environment.
        Args:
        action: The action to take (0-3 for directions)

        Returns:
        tuple: (observation, reward, terminated, truncated, info)
        """
	# Map the discrete action (0-3) to a movement direction
        direction = self._action_to_direction[action]

	# Update agent position, ensuring it stays within grid bounds
	# np.clip prevents the agent from walking off the edge
        self._agent_location = np.clip(self._agent_location + direction,0,self.size - 1)
        # self._agent_location = np.clip(self._agent_location + direction, 0, self.size - 1)
            
	# Check if agent reached the target
        terminated = np.array_equal(self._agent_location, self._target_location)

	# We don't use truncation in this simple environment
    # This means that the game will continue until A reaches T.
	# (could add a step limit here if desired)
        truncated = False

	# Simple reward structure: +1 for reaching target, 0 otherwise
	# Alternative: could give small negative rewards for each step to encourage efficiency
        reward = 1 if terminated else -0.1

        observation = self._get_obs()
        info = self._get_info()

        return observation, reward, terminated, truncated, info

    def render(self):
        grid = [ ["." for _ in range(self.size)] for _ in range(self.size)]
        if np.array_equal(self._target_location, self._agent_location):
            grid[self.size-1-self._target_location[1]] [self._target_location[0]] = 'V'
        else:
            grid[self.size-1-self._target_location[1]] [self._target_location[0]] = 'T'
            grid[self.size-1-self._agent_location[1]] [self._agent_location[0]] = 'A'
        lines = [" ".join(row) for row in grid]
        return "\n".join(lines)

## State Encoding

Q-Learning maintains a table of Q-values indexed by `(state, action)` pairs. Because observations are Python dictionaries, they cannot be used as dictionary keys directly. `encode_obs` converts an observation into a flat tuple `(ax, ay, tx, ty)` that is hashable and compact.

The `GridAgent` class wraps the Q-table and implements:
- **ε-greedy action selection** — random exploration with probability ε, greedy exploitation otherwise.
- **Q-value update** — the standard temporal-difference (TD) rule.
- **Epsilon decay** — ε is reduced linearly over training so the agent gradually shifts from exploration to exploitation.

In [ ]:
# Convert the observation dictionary into a hashable tuple.
# This tuple becomes the state used by the Q-table.
    
def encode_obs(obs):
    ax, ay = map(int, obs["agent"])
    tx, ty = map(int, obs["target"])
    return (ax, ay, tx, ty)

class GridAgent:
    def __init__(self, env, learning_rate, initial_epsilon, epsilon_decay,
                 final_epsilon, discount_factor=0.95):
        """Initialize a Reinforcement Learning agent with an empty dictionary
        of state-action values (q_values), a learning rate and an epsilon.

        Args:
            learning_rate: The learning rate
            initial_epsilon: The initial epsilon value
            epsilon_decay: The decay for epsilon
            final_epsilon: The final epsilon value
            discount_factor: The discount factor for computing the Q-value
        """
        self.q_values = defaultdict(lambda: np.zeros(env.action_space.n))

        self.lr = learning_rate
        self.discount_factor = discount_factor

        self.epsilon = initial_epsilon
        self.epsilon_decay = epsilon_decay
        self.final_epsilon = final_epsilon

        self.training_error = []

    def get_action(self, env, obs):
        """
        Returns the best action with probability (1 - epsilon)
        otherwise a random action with probability epsilon to ensure exploration.
        """
        # with probability epsilon return a random action to explore the environment
        if np.random.random() < self.epsilon:
            return env.action_space.sample()
        # with probability (1 - epsilon) act greedily (exploit)
        else:
            s = encode_obs(obs)
            return int(np.argmax(self.q_values[s]))
    
    def update(self, obs, action, reward, terminated, next_obs):
        """Updates the Q-value of an action."""

        s = encode_obs(obs)
        next_s = encode_obs(next_obs)
        future_q_value = (not terminated) * np.max(self.q_values[next_s])
        temporal_difference = (
            reward + self.discount_factor * future_q_value - self.q_values[s][action]
        )

        self.q_values[s][action] = (
            self.q_values[s][action] + self.lr * temporal_difference
        )
        self.training_error.append(temporal_difference)

    def decay_epsilon(self):
        self.epsilon = max(self.final_epsilon, self.epsilon - self.epsilon_decay)

## Training Loop

We now instantiate the environment and agent, then run the training loop.

**Hyperparameters: Turn this knobs**
| Parameter | Value | Role |
|---|---|---|
| `learning_rate` | 0.1 | Step size for Q-value updates |
| `n_episodes` | 100 000 | Total number of training episodes |
| `start_epsilon` | 1.0 | Initial exploration rate (fully random) |
| `final_epsilon` | 0.1 | Minimum exploration rate |
| `epsilon_decay` | `start / (n/2)` | Linear schedule: reaches minimum halfway through training |
| `discount_factor` | 0.95 | Weight given to future rewards |


The `playtrain` function runs a full episode. 

In [ ]:
from IPython.display import display, clear_output
 
def set_cursor(state):
    sys.stdout.write("\x1b[?25h" if state else "\x1b[?25l")

def clear_screen():
    sys.stdout.write("\x1b[2J\x1b[H")
    
def print_state(header, body):
    clear_screen()
    sys.stdout.write(header + '\n')
    sys.stdout.write(env.render())

def playtrain(render=False):
    obs, info = env.reset()
    done = False
    cnt = 0

    if render:
        set_cursor(False)

    while not done:
        action = agent.get_action(env, obs)

        if render:
            clear_output(wait=True)
            print_state(f'Move #{cnt}', env.render())
            sys.stdout.write('\n\n')
            if action == 0:
                sys.stdout.write('  .  \n')
                sys.stdout.write('. . R\n')
                sys.stdout.write('  .  \n')
            elif action == 1:
                sys.stdout.write('  U  \n')
                sys.stdout.write('. . .\n')
                sys.stdout.write('  .  \n')
            elif action == 2:
                sys.stdout.write('  .  \n')
                sys.stdout.write('L . .\n')
                sys.stdout.write('  .  \n')
            elif action == 3:
                sys.stdout.write('  .  \n')
                sys.stdout.write('. . .\n')
                sys.stdout.write('  D  \n')
            sys.stdout.flush()

        next_obs, reward, terminated, truncated, info = env.step(action)
        # update the agent
        agent.update(obs, action, reward, terminated, next_obs)

        if render:
                sys.stdout.write(f"Distance: {info['distance']:.2f}\n")
                #sys.stdout.flush()
                time.sleep(1)

        # update if the environment is done and the current obs
        done = terminated or truncated
        obs = next_obs
        cnt += 1

    agent.decay_epsilon()

    if render:
        clear_output(wait=True)
        print_state(f'Final state after {cnt} moves', env.render())
        set_cursor(True)

In [ ]:
env = GridWorldEnv(5)

# hyperparameters
learning_rate = 0.1
n_episodes = 100_000
start_epsilon = 1
epsilon_decay = start_epsilon / (n_episodes / 2)  # reduce the exploration over time
final_epsilon = 0.1

agent = GridAgent(
    env=env,
    learning_rate=learning_rate,
    initial_epsilon=start_epsilon,
    epsilon_decay=epsilon_decay,
    final_epsilon=final_epsilon,
)

set_cursor(False)
for episode in range(n_episodes):
    if not (episode % 100):
        clear_screen()
        print(f'EPISODE {episode} (eps = {agent.epsilon:.2f}, training error = {agent.training_error[-1] if episode else 0:.2f})')
    playtrain()

set_cursor(True)

## Visualising the Trained Agent

We run a single rendered episode to inspect the learned policy visually.

**Legend:**
| Symbol | Meaning |
|---|---|
| `A` | Agent |
| `T` | Target |
| `V` | Agent on target (episode complete) |
| `.` | Empty cell |

In [ ]:
playtrain(True)

## Questions

Think about these before moving on:

1. How many episodes are needed before the agent reliably reaches the target?
2. How many episodes are needed before the policy is near-optimal?
3. How does grid size affect training time?
4. How does truncation (stopping an episode if it takes too long) affect training for grids of different sizes? (search for 'truncated' in the code)

# Part 2 — Blind GridWorld

The agent can no longer observe the target's position. It only knows where it is on the grid.

This is a much harder problem. In a square grid the agent has no directional information about the target, so a purely random walk is the best it can do without additional structure. To make learning possible we instead use a **narrow rectangular grid** (`n=1` or `n=2` rows), where there is effectively only one dimension of freedom.

**Changes from Part 1:**
- `GridWorldEnv` now accepts `width` and `height` separately.
- The observation dictionary only contains `"agent"` — the `"target"` key is removed.
- `encode_obs` now returns `(ax, ay)` instead of `(ax, ay, tx, ty)`, halving the state-space size.

In [ ]:
def grid_distance(v, w, p=2):
    return np.linalg.norm(v - w, ord=p)

# Euclidean distance between two vectors.
# The parameter p controls the norm:
# p = 2 -> Euclidean norm
# p = 1 -> Manhattan norm


In [ ]:
# Define GridWorld and GridAgent

# GridWorld environment definition.
# The environment is a square grid where:
# - A = agent
# - T = target
# The goal is to learn a policy that reaches the target efficiently.

class GridWorldEnv(gym.Env):
    metadata = {"render_modes": ["ansi"], "render_fps": 4}
    
    def __init__(self, width, height):
        # The size of the square grid 
        self.width = width
        self.height = height

        # Initialize positions - will be set randomly in reset()
        # Using -1,-1 as "uninitialized" state
        self._agent_location = np.array([-1, -1], dtype=np.int32)
        self._target_location = np.array([-1, -1], dtype=np.int32)

        # Define what the agent can observe 
        # In this case we comment out the position of the target
        self.observation_space = gym.spaces.Dict(
                {
                    "agent" : gym.spaces.Box(low=np.array([0, 0]), high=np.array([self.width - 1, self.height - 1]),shape=(2,),dtype=int),   # [x, y] coordinates
                    # "target": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),  # [x, y] coordinates
                }
        )

        # Define what actions are available (4 directions)
        self.action_space = gym.spaces.Discrete(4)

        self.render_mode = "ansi"

        # Map action numbers to actual movements on the grid
        self._action_to_direction = {
                0: np.array([1, 0]),   # Move right (positive x)
                1: np.array([0, 1]),   # Move up (positive y)
                2: np.array([-1, 0]),  # Move left (negative x)
                3: np.array([0, -1]),  # Move down (negative y)
        }

    def _get_obs(self):
        """Convert internal state to observation format.
        Returns:
        dict: Observation with agent and target positions
        """
        return {"agent": self._agent_location}
        #return {"agent": self._agent_location, "target": self._target_location}

    
    def _get_info(self):
        distance = np.linalg.norm(self._agent_location - self._target_location,ord=2)
        return {"distance": distance}


    def reset(self, seed=None, options=None):
        """Start a new episode.
        Args:
        seed: Random seed for reproducible episodes
        options: Additional configuration (unused in this example)

        Returns:
        tuple: (observation, info) for the initial state
        """
        # IMPORTANT: Must call this first to seed the random number generator
        super().reset(seed=seed)

        # Randomly place the agent anywhere on the grid
        self._agent_location = np.array([self.np_random.integers(0, self.width),self.np_random.integers(0, self.height)], dtype=int)
            
        # Randomly place target, ensuring it's different from agent position
        self._target_location = self._agent_location
        while np.array_equal(self._target_location, self._agent_location):
            self._target_location = np.array([self.np_random.integers(0, self.width),self.np_random.integers(0, self.height)], dtype=int)
        observation = self._get_obs()
        info = self._get_info()
        return observation, info
	
    def step(self, action):
        """Execute one timestep within the environment.
        Args:
        action: The action to take (0-3 for directions)

        Returns:
        tuple: (observation, reward, terminated, truncated, info)
        """
	# Map the discrete action (0-3) to a movement direction
        direction = self._action_to_direction[action]

	# Update agent position, ensuring it stays within grid bounds
	# np.clip prevents the agent from walking off the edge
        #self._agent_location = np.clip(self._agent_location + direction,0,self.size - 1)
        self._agent_location = self._agent_location + direction
        self._agent_location[0] = np.clip(self._agent_location[0],0,self.width - 1)
        self._agent_location[1] = np.clip(self._agent_location[1],0, self.height - 1)
        
        # self._agent_location = np.clip(self._agent_location + direction, 0, self.size - 1)
            
	# Check if agent reached the target
        terminated = np.array_equal(self._agent_location, self._target_location)

	# We don't use truncation in this simple environment
    # This means that the game will continue until A reaches T.
	# (could add a step limit here if desired)
        truncated = False

	# Simple reward structure: +1 for reaching target, 0 otherwise
	# Alternative: could give small negative rewards for each step to encourage efficiency
        reward = 1 if terminated else -0.1

        observation = self._get_obs()
        info = self._get_info()

        return observation, reward, terminated, truncated, info

    def render(self):
        grid = [ ["." for _ in range(self.width)] for _ in range(self.height)]
        if np.array_equal(self._target_location, self._agent_location):
            grid[self.height - 1 - self._target_location[1]][self._target_location[0]] = 'V'
        else:
            grid[self.height - 1 - self._target_location[1]][self._target_location[0]] = 'T'
            grid[self.height - 1 - self._agent_location[1]][self._agent_location[0]] = 'A'
        lines = [" ".join(row) for row in grid]
        return "\n".join(lines)

In [ ]:
# Convert the observation dictionary into a hashable tuple.
# This tuple becomes the state used by the Q-table.
    
def encode_obs(obs):
    ax, ay = map(int, obs["agent"])
    #tx, ty = map(int, obs["target"])
    #return (ax, ay, tx, ty)
    return(ax, ay)

class GridAgent:
    def __init__(self, env, learning_rate, initial_epsilon, epsilon_decay,
                 final_epsilon, discount_factor=0.95):
        """Initialize a Reinforcement Learning agent with an empty dictionary
        of state-action values (q_values), a learning rate and an epsilon.

        Args:
            learning_rate: The learning rate
            initial_epsilon: The initial epsilon value
            epsilon_decay: The decay for epsilon
            final_epsilon: The final epsilon value
            discount_factor: The discount factor for computing the Q-value
        """
        self.q_values = defaultdict(lambda: np.zeros(env.action_space.n))

        self.lr = learning_rate
        self.discount_factor = discount_factor

        self.epsilon = initial_epsilon
        self.epsilon_decay = epsilon_decay
        self.final_epsilon = final_epsilon

        self.training_error = []

    def get_action(self, env, obs):
        """
        Returns the best action with probability (1 - epsilon)
        otherwise a random action with probability epsilon to ensure exploration.
        """
        # with probability epsilon return a random action to explore the environment
        if np.random.random() < self.epsilon:
            return env.action_space.sample()
        # with probability (1 - epsilon) act greedily (exploit)
        else:
            s = encode_obs(obs)
            return int(np.argmax(self.q_values[s]))
    
    def update(self, obs, action, reward, terminated, next_obs):
        """Updates the Q-value of an action."""

        s = encode_obs(obs)
        next_s = encode_obs(next_obs)
        future_q_value = (not terminated) * np.max(self.q_values[next_s])
        temporal_difference = (
            reward + self.discount_factor * future_q_value - self.q_values[s][action]
        )

        self.q_values[s][action] = (
            self.q_values[s][action] + self.lr * temporal_difference
        )
        self.training_error.append(temporal_difference)

    def decay_epsilon(self):
        self.epsilon = max(self.final_epsilon, self.epsilon - self.epsilon_decay)

In [ ]:
from IPython.display import display, clear_output

def set_cursor(state):
    sys.stdout.write("\x1b[?25h" if state else "\x1b[?25l")

def clear_screen():
    sys.stdout.write("\x1b[2J\x1b[H")
    
def print_state(header, body):
    clear_screen()
    sys.stdout.write(header + '\n')
    sys.stdout.write(env.render())


def playtrain(render=False):
    obs, info = env.reset()
    done = False
    cnt = 0

    if render:
        set_cursor(False)

    while not done:
        action = agent.get_action(env, obs)

        if render:
            clear_output(wait=True)
            print_state(f'Move #{cnt}', env.render())
            sys.stdout.write('\n\n')
            if action == 0:
                sys.stdout.write('  .  \n')
                sys.stdout.write('. . R\n')
                sys.stdout.write('  .  \n')
            elif action == 1:
                sys.stdout.write('  U  \n')
                sys.stdout.write('. . .\n')
                sys.stdout.write('  .  \n')
            elif action == 2:
                sys.stdout.write('  .  \n')
                sys.stdout.write('L . .\n')
                sys.stdout.write('  .  \n')
            elif action == 3:
                sys.stdout.write('  .  \n')
                sys.stdout.write('. . .\n')
                sys.stdout.write('  D  \n')
            sys.stdout.flush()

        next_obs, reward, terminated, truncated, info = env.step(action)
        # update the agent
        agent.update(obs, action, reward, terminated, next_obs)

        if render:
                sys.stdout.write(f"Distance: {info['distance']:.2f}\n")
                #sys.stdout.flush()
                time.sleep(1)

        # update if the environment is done and the current obs
        done = terminated or truncated
        obs = next_obs
        cnt += 1

    agent.decay_epsilon()

    if render:
        clear_output(wait=True)
        print_state(f'Final state after {cnt} moves', env.render())
        set_cursor(True)

In [ ]:
env = GridWorldEnv(2,5)

# hyperparameters
learning_rate = 0.1
n_episodes = 100_000
start_epsilon = 1
epsilon_decay = start_epsilon / (n_episodes / 2)  # reduce the exploration over time
final_epsilon = 0.1

agent = GridAgent(
    env=env,
    learning_rate=learning_rate,
    initial_epsilon=start_epsilon,
    epsilon_decay=epsilon_decay,
    final_epsilon=final_epsilon,
)

set_cursor(False)
for episode in range(n_episodes):
    if not (episode % 100):
        clear_screen()
        print(f'EPISODE {episode} (eps = {agent.epsilon:.2f}, training error = {agent.training_error[-1] if episode else 0:.2f})')
    playtrain()

set_cursor(True)

In [ ]:
playtrain(True)

# Questions 

Think about these before moving on:

1. How does knowing the position of `T`effect the learning? 
2. What do you think is the strategy of `A`?

# Part 3 — Toroidal GridWorld

Here we change the geometry of the environment.

Instead of walls, the grid wraps around like a torus:
- moving right off the last column reappears at column 0.
- moving up off the last row reappears at row 0.

This has two consequences:
- **Movement** is updated using modular arithmetic.
- **Distance** can no longer be computed with a simple Euclidean norm — we must account for all four "wrapped" translations and take the minimum.

Full observability is restored: the agent again knows both positions.

In [ ]:
# Distance on a torus.
# We consider all translated copies of the grid and take the minimum distance.


def torus_distance(v, w, size, p=2):
    translations = [np.array(foo) for foo in [ [0,0], [size,0], [0,size], [size,size]]]
    return min(np.linalg.norm(v - w + t, ord=p) for t in translations)

In [ ]:
# Your code here